# Lesson 10: Map-Reduce on a List of Numbers

## Objective
Apply a **map-reduce** pattern: fan-out to process each number in a list independently (map phase), then aggregate all results into a final sum (reduce phase).

## Limitation of Lesson 9
- ❌ Lesson 9 ran fixed parallel branches (sum + product in parallel)
- ❌ No way to handle a **variable-length** list of inputs
- ❌ No dynamic fan-out based on data

## What is NEW in Lesson 10?
- ✅ **Dynamic fan-out**: spawn one subgraph per input item
- ✅ **`Send` API**: the LangGraph primitive for dynamic parallel dispatch
- ✅ **Map phase**: process each number independently
- ✅ **Reduce phase**: aggregate results using a reducer
- ✅ Variable-length parallelism

## Theory: Send API & Map-Reduce

```
           ┌── process_item(num=2) ──┐
           ├── process_item(num=5) ──┤
START ─────┼── process_item(num=7) ──┼──→ aggregate → END
           ├── process_item(num=1) ──┤
           └── process_item(num=9) ──┘
```

`Send(node_name, state_update)` tells LangGraph:
> "Dispatch execution to `node_name` with this specific state, in parallel"

The reduce phase uses a **reducer** (Lesson 4) to aggregate all partial results into one list.


In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from typing import TypedDict, Annotated
from IPython.display import Image, display

# Send: the key API for dynamic fan-out (map phase)
print("✓ Imports — key new import: Send from langgraph.types")


In [ ]:
# ── Cell 2: State Schemas ────────────────────────────────────────────────────
# We need TWO state schemas:
# 1. OverallState — the main graph state (holds the list + accumulated results)
# 2. ItemState    — the per-item state for the map worker nodes

class OverallState(TypedDict):
    numbers: list[int]                       # Input: list of numbers to process
    squared: Annotated[list, operator.add]   # Accumulate: each item's square
    total: float                             # Final sum of all squares

class ItemState(TypedDict):
    number: int     # A single number to process

print("✓ OverallState and ItemState defined")
print("  OverallState.squared uses operator.add reducer (appends)")
print("  ItemState is the per-item worker state")


In [ ]:
# ── Cell 3: Map Phase — Fan-Out Router ───────────────────────────────────────
# This function is called as a conditional edge source
# It returns a LIST of Send objects — one per input number
# LangGraph dispatches ALL of them in parallel

def fan_out(state: OverallState) -> list[Send]:
    """
    Map phase: for each number in the list, dispatch a Send to process_item.
    LangGraph runs all Send() calls in parallel.
    """
    sends = []
    for num in state["numbers"]:
        sends.append(
            Send("process_item", {"number": num})
            # ↑ node name    ↑ state for that specific worker
        )
    print(f"  [fan_out] Dispatching {len(sends)} parallel workers")
    return sends

print("✓ fan_out defined")
print("  Returns List[Send] — one Send per number in the list")


In [ ]:
# ── Cell 4: Map Worker Node ──────────────────────────────────────────────────
# This node runs ONCE PER NUMBER — in parallel for all numbers
# Returns a partial OverallState update (only the 'squared' reducer field)

def process_item(state: ItemState) -> dict:
    num = state["number"]
    squared = num ** 2
    print(f"  [process_item] {num}² = {squared}")
    return {"squared": [squared]}  # Must be a list for the reducer

print("✓ process_item (map worker) defined")


In [ ]:
# ── Cell 5: Reduce Phase — Aggregate Results ────────────────────────────────
def aggregate(state: OverallState) -> dict:
    total = sum(state["squared"])
    print(f"  [aggregate] Squared values: {state['squared']}")
    print(f"  [aggregate] Sum of squares: {total}")
    return {"total": float(total)}

print("✓ aggregate (reduce phase) defined")


In [ ]:
# ── Cell 6: Build Map-Reduce Graph ───────────────────────────────────────────
builder = StateGraph(OverallState)

builder.add_node("process_item", process_item)
builder.add_node("aggregate",    aggregate)

# Fan-out: START → fan_out router → N×process_item (parallel)
# fan_out returns List[Send] → LangGraph dispatches them all
builder.add_conditional_edges(START, fan_out, ["process_item"])

# Fan-in: after ALL process_item nodes complete → aggregate
builder.add_edge("process_item", "aggregate")
builder.add_edge("aggregate",    END)

graph = builder.compile()
print("✓ Map-reduce graph compiled")


In [ ]:
# ── Cell 7: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 8: Run with a list of numbers ───────────────────────────────────────
numbers = [1, 2, 3, 4, 5]
print(f"Input numbers: {numbers}")
print(f"Expected: sum of squares = {sum(x**2 for x in numbers)}")
print("-" * 45)

state = {"numbers": numbers, "squared": [], "total": 0.0}
out = graph.invoke(state)

print("-" * 45)
print(f"\n  Squared values : {out['squared']}")
print(f"  Sum of squares : {out['total']}")


In [ ]:
# ── Cell 9: Test with different lists ────────────────────────────────────────
test_cases = [
    [2, 4, 6, 8, 10],
    [3, 3, 3],
    [1],
    [5, 12, 13],   # Pythagorean triple
]

print(f"{'Numbers':<25} {'Sum of Squares':>15} {'Expected':>10} {'Match':>6}")
print("-" * 60)

for nums in test_cases:
    state = {"numbers": nums, "squared": [], "total": 0.0}
    out = graph.invoke(state)
    expected = sum(x**2 for x in nums)
    match = "✓" if out['total'] == expected else "✗"
    print(f"{str(nums):<25} {out['total']:>15.0f} {expected:>10} {match:>6}")


## The `Send` API — Deep Dive

```python
Send("node_name", state_update_dict)
```

- Tells LangGraph: "run `node_name` with this state update as its input"
- All `Send`s returned from a single router are dispatched **in parallel**
- Each worker receives its own isolated copy of `ItemState`
- Results are merged back into `OverallState` via the **reducer**

### Why the reducer is critical here
Without `Annotated[list, operator.add]`:
- Each parallel `process_item` returns `{"squared": [X]}`
- The last one to complete would OVERWRITE all others
- Result would be `[last_item_squared]` instead of all items

With the reducer:
- Each `process_item`'s `[X]` is **appended** to `squared`
- Final `squared` = all items' squares, regardless of completion order

## Summary

| Phase | Node | What it does |
|-------|------|-------------|
| Map | `fan_out` | Returns `List[Send]` — dispatches N workers in parallel |
| Map | `process_item` | Processes one number, returns partial update |
| Reduce | `aggregate` | Receives all partial results (via reducer), computes final answer |

## Why This Matters in Production
- **Batch LLM calls**: process 100 documents in parallel
- **Parallel API calls**: call N external services simultaneously
- **Distributed computation**: split large work into independent chunks

## Limitations → What Lesson 11 Solves
- ❌ All logic is in one flat graph — hard to reuse
- ❌ No modularity — you can't embed this arithmetic engine inside another graph
- ❌ **Lesson 11**: Subgraphs — compile a graph and use it AS a node inside another graph
